In [1]:
from typing import Union, Callable, Any, Optional

import torch
from torch import nn
import torch.nn.functional as F
from torch.nn.modules.module import T
from torch.utils.hooks import RemovableHandle

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


### Things to do:
- pad with zeros to max context?
    - I guess not - the only dependence on sequence length is the qk multiplication, which is multiplied out by the end

In [4]:
class SelfAttention(nn.Module):
    def __init__(self, embedding_dim=768, qk_dim=128, max_context=100):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.qk_dim = qk_dim
        self.max_context = max_context

        self.sqrt_qk_dim = qk_dim ** 0.5

        self.query_embed = nn.Linear(embedding_dim, qk_dim)
        self.key_embed = nn.Linear(embedding_dim, qk_dim)

        self.value_matrix = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, x):
        x = x[:,-self.max_context:,:]
        q = self.query_embed(x)
        k = self.key_embed(x)
        v = self.value_matrix(x)
        attn_scores = q @ k.mT / self.sqrt_qk_dim

        attn_probs = torch.softmax(attn_scores, dim=-1)

        return attn_probs @ v

In [5]:
class LowRankSelfAttention(nn.Module):
    def __init__(self, embedding_dim=768, qk_dim=128, max_context=100, low_rank=True):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.qk_dim = qk_dim
        self.max_context = max_context

        self.sqrt_qk_dim = qk_dim ** 0.5

        self.query_embed = nn.Linear(embedding_dim, qk_dim)
        self.key_embed = nn.Linear(embedding_dim, qk_dim)

        self.value_matrix = nn.Sequential(nn.Linear(embedding_dim, qk_dim), nn.Linear(qk_dim, embedding_dim))

    def forward(self, x):
        x = x[:,-self.max_context:,:]
        q = self.query_embed(x)
        k = self.key_embed(x)
        v = self.value_matrix(x)
        attn_scores = q @ k.mT / self.sqrt_qk_dim

        attn_probs = torch.softmax(attn_scores, dim=-1)

        return attn_probs @ v

In [6]:
class MaskedSelfAttention(nn.Module):
    def __init__(self, embedding_dim=768, qk_dim=128, mask=True, max_context=100):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.qk_dim = qk_dim
        self.max_context = max_context
        self.mask = mask

        self.sqrt_qk_dim = qk_dim ** 0.5

        self.query_embed = nn.Linear(embedding_dim, qk_dim)
        self.key_embed = nn.Linear(embedding_dim, qk_dim)

        self.value_matrix = nn.Sequential(nn.Linear(embedding_dim, qk_dim), nn.Linear(qk_dim, embedding_dim))

        if mask:
            self.register_buffer("mask_array", torch.tril(torch.ones(max_context, max_context, dtype=torch.bool)))

    def forward(self, x):
        x = x[:,-self.max_context:,:]
        q = self.query_embed(x)
        k = self.key_embed(x)
        v = self.value_matrix(x)
        attn_scores = q @ k.mT / self.sqrt_qk_dim

        if self.mask:
            attn_scores = attn_scores.masked_fill(self.mask_array[:x.shape[1], :x.shape[1]] == 0, float("-inf"))

        attn_probs = torch.softmax(attn_scores, dim=-1)

        return attn_probs @ v





In [7]:
# Introduce multiple heads and merge q and k layers for speed

class MaskedMultiHeadSelfAttention(nn.Module):
    def __init__(self, embedding_dim=768, qk_dim=128, causal=True, max_context=100, n_heads=8, head_dim=None):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.qk_dim = qk_dim
        self.max_context = max_context
        self.causal = causal
        self.n_heads = n_heads

        if head_dim is None:
            self.head_dim = embedding_dim // n_heads
        else:
            self.head_dim = head_dim

        self.sqrt_qk_dim = qk_dim ** 0.5

        self.qk = nn.Linear(embedding_dim, 2 * qk_dim * n_heads)

        self.value_matrix = nn.Sequential(nn.Linear(embedding_dim, n_heads * qk_dim), nn.Linear(n_heads * qk_dim, self.head_dim * self.n_heads))

        if causal:
            self.register_buffer("mask_array", torch.tril(torch.ones(max_context, max_context, dtype=torch.bool)))

    def forward(self, x):
        batch_dim, seq_len, embed_dim = x.shape
        x = x[:,-self.max_context:,:]
        qk = self.qk(x)
        q, k = torch.chunk(qk, 2, dim=-1) # batch, seq, n_heads * qk_dim

        # Use view to prevent copying
        q = q.view(batch_dim, seq_len, self.n_heads, self.qk_dim).transpose(1,2) # batch, n_heads, seq, qk_dim
        k_mT = k.view(batch_dim, seq_len, self.n_heads, self.qk_dim).permute(0,2,3,1) # batch, n_heads, qk_dim, seq


        v = self.value_matrix(x).view(batch_dim, seq_len, self.n_heads, self.head_dim).transpose(1,2) # batch, n_heads, seq, head_dim
        attn_scores = q @ k_mT / self.sqrt_qk_dim # batch, n_heads, seq, seq

        if self.causal:
            attn_scores = attn_scores.masked_fill(self.mask_array[:seq_len, :seq_len] == 0, float("-inf"))

        attn_probs = torch.softmax(attn_scores, dim=-1) # batch, n_heads, seq, seq

        embedding_updates = attn_probs @ v # batch, n_heads, seq, head_dim
        #TODO: this reshape copies data. view would be better if I can get the shapes to work.
        return embedding_updates.transpose(1,2).reshape(batch_dim, seq_len, self.head_dim * self.n_heads) # batch, seq, n_heads * head_dim

In [8]:
# use inbuilt scaled dot product attention for speed
# This doesn't seem to be faster in my case - either on my laptop or the box.

class MaskedMultiHeadSDPASelfAttention(nn.Module):
    def __init__(self, embedding_dim=768, qk_dim=128, causal=True, max_context=100, n_heads=8, head_dim=None):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.qk_dim = qk_dim
        self.max_context = max_context
        self.causal = causal
        self.n_heads = n_heads

        if head_dim is None:
            self.head_dim = embedding_dim // n_heads
        else:
            self.head_dim = head_dim

        self.sqrt_qk_dim = qk_dim ** 0.5

        self.qk = nn.Linear(embedding_dim, 2 * qk_dim * n_heads)

        self.value_matrix = nn.Sequential(nn.Linear(embedding_dim, n_heads * qk_dim), nn.Linear(n_heads * qk_dim, self.head_dim * self.n_heads))



    def forward(self, x):
        batch_dim, seq_len, embed_dim = x.shape
        x = x[:,-self.max_context:,:]
        qk = self.qk(x)
        q, k = torch.chunk(qk, 2, dim=-1) # batch, seq, n_heads * qk_dim

        # Use view to prevent copying
        q = q.view(batch_dim, seq_len, self.n_heads, self.qk_dim).transpose(1,2) # batch, n_heads, seq, qk_dim
        k = k.view(batch_dim, seq_len, self.n_heads, self.qk_dim).transpose(1,2) # batch, n_heads, qk_dim, seq


        v = self.value_matrix(x).view(batch_dim, seq_len, self.n_heads, self.head_dim).transpose(1,2) # batch, n_heads, seq, head_dim
        attn_output = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal)

        #TODO: this reshape copies data. view would be better if I can get the shapes to work.
        return attn_output.transpose(1,2).reshape(batch_dim, seq_len, self.head_dim * self.n_heads) # batch, seq, n_heads * head_dim

In [9]:
# Implement deepseek v2 multihead latent attention
# Q remains full rank, both k and v are compressed. The compression is shared between k, v, and over all heads - one down matrix for all k and v heads.

class MaskedMultiHeadLatentSelfAttention(nn.Module):
    def __init__(self, embedding_dim=384, projection_dim=128, qk_dim=64, v_dim=64, causal=True, max_context=100, n_heads=6):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.qk_dim = qk_dim
        self.v_dim = v_dim
        self.projection_dim = projection_dim
        self.max_context = max_context
        self.causal = causal
        self.n_heads = n_heads

        self.sqrt_qk_dim = qk_dim ** 0.5

        # Want the total number of parameters for kv to be = to the number of parameters for q.
        # q_params = d_embed * d_qk * n_heads
        # kv_params = d_embed * d_proj + d_proj * (d_qk + d_v) * n_heads
        # In the previous layers, I speparated d_qk and dv, letting n_heads * d_v = d_embed. This means that the concatenation of all heads injects a full embedding's worth of information
        # After reading a bit more, it might make sense for d_v = d_qk = d_embed / n_heads. This means that each attention head attends to 1/n_heads worth of embedding, then also injects ~ 1/n_heads embeddings worth of information. Nice and neat. Clarification needed on the empirically optimal choice of these dimensions, as well as the relative balance of parameter counts between q, k, and v.

        # Taking d_v = d_qk, and setting q_params = kv_params, we get:
        # d_embed * d_qk * n_heads = d_embed * d_proj + d_proj * 2 * d_qk * n_heads
        # d_proj = d_embed * d_qk * n_heads / (d_embed + 2 * d_qk * n_heads)
        # if d_qk = d_embed / n_heads:
        # d_proj = d_embed / 3

        # For the default parameters, let embedding_dim = 384, n_heads = 6
        # d_qk = 384 / 6 = 64
        # d_v = d_qk = 384 / 6 = 64
        # d_proj = 384 / 3 = 128

        self.q = nn.Linear(embedding_dim, qk_dim * n_heads) # Grouped matrix for the heads
        self.kv = nn.Sequential(nn.Linear(embedding_dim, projection_dim), nn.Linear(projection_dim, (qk_dim + v_dim) * n_heads)) # Grouped matrix for the heads and k and v

        if causal:
            self.register_buffer("mask_array", torch.tril(torch.ones(max_context, max_context, dtype=torch.bool)))

    def forward(self, x):
        batch_dim, seq_len, embed_dim = x.shape
        x = x[:, -self.max_context:, :]
        q = self.q(x) # batch, seq, n_heads * qk_dim

        kv = self.kv(x) # batch, seq, n_heads * (qk_dim + v_dim)
        k, v = torch.split(kv, [self.qk_dim * self.n_heads, self.v_dim * self.n_heads], dim=-1) # batch, seq, n_heads * qk_dim; batch, seq, n_heads * v_dim

        # Use view to prevent copying
        q = q.view(batch_dim, seq_len, self.n_heads, self.qk_dim).transpose(1, 2)  # batch, n_heads, seq, qk_dim
        k_mT = k.view(batch_dim, seq_len, self.n_heads, self.qk_dim).permute(0, 2, 3, 1)  # batch, n_heads, qk_dim, seq
        v = v.view(batch_dim, seq_len, self.n_heads, self.v_dim).transpose(1,2) # batch, n_heads, seq, v_dim

        attn_scores = q @ k_mT / self.sqrt_qk_dim  # batch, n_heads, seq, seq

        if self.causal:
            attn_scores = attn_scores.masked_fill(self.mask_array[:seq_len, :seq_len] == 0, float("-inf"))

        attn_probs = torch.softmax(attn_scores, dim=-1)  # batch, n_heads, seq, seq

        embedding_updates = attn_probs @ v  # batch, n_heads, seq, v_dim
        #TODO: this reshape copies data. view would be better if I can get the shapes to work.
        return embedding_updates.transpose(1, 2).reshape(batch_dim, seq_len,
                                                         self.v_dim * self.n_heads)  # batch, seq, n_heads * v_dim -> output dimension = n_heads * v_dim (= embedding_dim by default)

In [10]:
# Implement deepseek v2 multihead latent attention
# Q remains full rank, both k and v are compressed. The compression is shared between k, v, and over all heads - one down matrix for all k and v heads.

class MaskedMultiHeadLatentSelfAttention4DOut(nn.Module):
    def __init__(self, embedding_dim=384, projection_dim=128, qk_dim=64, v_dim=64, causal=True, max_context=100, n_heads=6):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.qk_dim = qk_dim
        self.v_dim = v_dim
        self.projection_dim = projection_dim
        self.max_context = max_context
        self.causal = causal
        self.n_heads = n_heads

        self.sqrt_qk_dim = qk_dim ** 0.5

        # Want the total number of parameters for kv to be = to the number of parameters for q.
        # q_params = d_embed * d_qk * n_heads
        # kv_params = d_embed * d_proj + d_proj * (d_qk + d_v) * n_heads
        # In the previous layers, I speparated d_qk and dv, letting n_heads * d_v = d_embed. This means that the concatenation of all heads injects a full embedding's worth of information
        # After reading a bit more, it might make sense for d_v = d_qk = d_embed / n_heads. This means that each attention head attends to 1/n_heads worth of embedding, then also injects ~ 1/n_heads embeddings worth of information. Nice and neat. Clarification needed on the empirically optimal choice of these dimensions, as well as the relative balance of parameter counts between q, k, and v.

        # Taking d_v = d_qk, and setting q_params = kv_params, we get:
        # d_embed * d_qk * n_heads = d_embed * d_proj + d_proj * 2 * d_qk * n_heads
        # d_proj = d_embed * d_qk * n_heads / (d_embed + 2 * d_qk * n_heads)
        # if d_qk = d_embed / n_heads:
        # d_proj = d_embed / 3

        # For the default parameters, let embedding_dim = 384, n_heads = 6
        # d_qk = 384 / 6 = 64
        # d_v = d_qk = 384 / 6 = 64
        # d_proj = 384 / 3 = 128

        self.q = nn.Linear(embedding_dim, qk_dim * n_heads) # Grouped matrix for the heads
        self.kv = nn.Sequential(nn.Linear(embedding_dim, projection_dim), nn.Linear(projection_dim, (qk_dim + v_dim) * n_heads)) # Grouped matrix for the heads and k and v

        if causal:
            self.register_buffer("mask_array", torch.tril(torch.ones(max_context, max_context, dtype=torch.bool)))

    def forward(self, x):
        batch_dim, seq_len, embed_dim = x.shape
        x = x[:, -self.max_context:, :]
        q = self.q(x) # batch, seq, n_heads * qk_dim

        kv = self.kv(x) # batch, seq, n_heads * (qk_dim + v_dim)
        k, v = torch.split(kv, [self.qk_dim * self.n_heads, self.v_dim * self.n_heads], dim=-1) # batch, seq, n_heads * qk_dim; batch, seq, n_heads * v_dim

        # Use view to prevent copying
        q = q.view(batch_dim, seq_len, self.n_heads, self.qk_dim).transpose(1, 2)  # batch, n_heads, seq, qk_dim
        k_mT = k.view(batch_dim, seq_len, self.n_heads, self.qk_dim).permute(0, 2, 3, 1)  # batch, n_heads, qk_dim, seq
        v = v.view(batch_dim, seq_len, self.n_heads, self.v_dim).transpose(1,2) # batch, n_heads, seq, v_dim

        attn_scores = q @ k_mT / self.sqrt_qk_dim  # batch, n_heads, seq, seq

        if self.causal:
            attn_scores = attn_scores.masked_fill(self.mask_array[:seq_len, :seq_len] == 0, float("-inf"))

        attn_probs = torch.softmax(attn_scores, dim=-1)  # batch, n_heads, seq, seq
        embedding_updates = attn_probs @ v  # batch, n_heads, seq, v_dim
        #TODO: this reshape copies data. view would be better if I can get the shapes to work.
        return embedding_updates.transpose(1, 2).reshape(batch_dim, seq_len,
                                                         self.v_dim * self.n_heads)  # batch, seq, n_heads * v_dim -> output dimension = n_heads * v_dim (= embedding_dim by default)

In [11]:
test = MaskedMultiHeadLatentSelfAttention().to('cpu')
x = torch.rand(1000, 100, 384).to('cpu')
test.forward(x)
%timeit test.forward(x)
%timeit test.forward(x)
%timeit test.forward(x)
%timeit test.forward(x)

547 ms ± 103 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
557 ms ± 195 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
574 ms ± 160 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
595 ms ± 199 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
test = MaskedMultiHeadLatentSelfAttention().to(device)
x = x.to(device)
%timeit test.forward(x)
%timeit test.forward(x)
%timeit test.forward(x)
%timeit test.forward(x)

1.31 ms ± 350 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)
134 ms ± 1.74 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
135 ms ± 1.21 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
132 ms ± 414 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [13]:
test = MaskedMultiHeadLatentSelfAttention4DOut().to('cpu')
x = x.to('cpu')
%timeit test.forward(x)
%timeit test.forward(x)
%timeit test.forward(x)
%timeit test.forward(x)

459 ms ± 92.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
393 ms ± 21.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
397 ms ± 22.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
390 ms ± 12.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [14]:
test = MaskedMultiHeadLatentSelfAttention4DOut().to(device)
x = x.to(device)
%timeit test.forward(x)
%timeit test.forward(x)
%timeit test.forward(x)
%timeit test.forward(x)

133 ms ± 1.05 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
132 ms ± 157 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
133 ms ± 1.15 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
132 ms ± 716 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


# To Add:
- [ ] rope
- [ ] kv caching
- [ ] flashattention?
- [ ] layernorm or batchnorm or rmsnorm
- [ ] dropout
- [x] shared low-rank k and v? (deepseek)
- [ ] pytorch inbuilt methods

In [ ]:
class OldRoPETransform(nn.Module):
    def __init__(self, max_seq_len, dim, base=10000):
        super().__init__()
        m_values = torch.arange(max_seq_len, requires_grad=False)
        assert dim % 2 == 0, "RoPE requires even dimension"

        theta_values = torch.repeat_interleave(torch.pow(base, -2 * torch.arange(dim//2, requires_grad=False) / dim), 2)
        trig_args = torch.outer(m_values, theta_values) # max_seq_len, dim
        # I need to wrap these in a buffer or parameter so that they get moved to the correct device when the model is moved.
        cos_vals = torch.cos(trig_args)
        sin_vals = torch.sin(trig_args) * torch.tensor([[-1, 1]], requires_grad=False).repeat(max_seq_len, dim//2) # multiply by alternation -1 +1s for correct rotation application
        self.register_buffer("cos_vals", cos_vals)
        self.register_buffer("sin_vals", sin_vals)

    def forward(self, x):
        batch_dim, n_heads, seq, qk_dim = x.shape
        # x is batch, n_heads, seq, qk_dim
        # should be broadcast correctly?
        x_pairwise_swapped = x.reshape(batch_dim, n_heads, seq, 2, qk_dim//2)
        x_pairwise_swapped = torch.flip(x_pairwise_swapped, (3,)).reshape(batch_dim, n_heads, seq, qk_dim)
        x = self.cos_vals[-seq:,:] * x + self.sin_vals[-seq:,:] * x_pairwise_swapped
        return x



In [17]:
class RoPETransform(nn.Module):
    def __init__(self, max_seq_len, dim, base=10000):
        super().__init__()
        m_values = torch.arange(max_seq_len, requires_grad=False)
        assert dim % 2 == 0, "RoPE requires even dimension"

        theta_values = torch.repeat_interleave(torch.pow(base, -2 * torch.arange(dim//2, requires_grad=False) / dim), 2)
        trig_args = torch.outer(m_values, theta_values) # max_seq_len, dim
        # I need to wrap these in a buffer or parameter so that they get moved to the correct device when the model is moved.
        cos_vals = torch.cos(trig_args)
        sin_vals = torch.sin(trig_args)# multiply by alternation -1 +1s for correct rotation application
        self.register_buffer("cos_vals", cos_vals)
        self.register_buffer("sin_vals", sin_vals)

    def forward(self, x):
        batch_dim, n_heads, seq, qk_dim = x.shape
        # x is batch, n_heads, seq, qk_dim
        x1, x2 = x[...,::2], x[...,1::2]
        cos = self.cos_vals[-seq:].to(dtype=x.dtype, device=x.device)
        sin = self.sin_vals[-seq:].to(dtype=x.dtype, device=x.device)
        x_rotated = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)

        return x_rotated

class MaskedMultiHeadLatentRoPESelfAttention(nn.Module):
    def __init__(self, embedding_dim=384, projection_dim=128, qk_dim=64, v_dim=64, causal=True, max_context=100, n_heads=6, sdpa=False, dropout=0.1):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.qk_dim = qk_dim
        self.v_dim = v_dim
        self.projection_dim = projection_dim
        self.max_context = max_context
        self.causal = causal
        self.n_heads = n_heads
        self.sdpa = sdpa
        self.dropout = dropout
        self.dropout_layer = nn.Dropout(dropout)


        self.sqrt_qk_dim = qk_dim ** 0.5

        self.qk_rope = RoPETransform(max_context, qk_dim)
        #self.v_rope = RoPETransform(max_context, v_dim)

        # Want the total number of parameters for kv to be = to the number of parameters for q.
        # q_params = d_embed * d_qk * n_heads
        # kv_params = d_embed * d_proj + d_proj * (d_qk + d_v) * n_heads
        # In the previous layers, I speparated d_qk and dv, letting n_heads * d_v = d_embed. This means that the concatenation of all heads injects a full embedding's worth of information
        # After reading a bit more, it might make sense for d_v = d_qk = d_embed / n_heads. This means that each attention head attends to 1/n_heads worth of embedding, then also injects ~ 1/n_heads embeddings worth of information. Nice and neat. Clarification needed on the empirically optimal choice of these dimensions, as well as the relative balance of parameter counts between q, k, and v.

        # Taking d_v = d_qk, and setting q_params = kv_params, we get:
        # d_embed * d_qk * n_heads = d_embed * d_proj + d_proj * 2 * d_qk * n_heads
        # d_proj = d_embed * d_qk * n_heads / (d_embed + 2 * d_qk * n_heads)
        # if d_qk = d_embed / n_heads:
        # d_proj = d_embed / 3

        # For the default parameters, let embedding_dim = 384, n_heads = 6
        # d_qk = 384 / 6 = 64
        # d_v = d_qk = 384 / 6 = 64
        # d_proj = 384 / 3 = 128

        self.q = nn.Linear(embedding_dim, qk_dim * n_heads) # Grouped matrix for the heads
        self.kv = nn.Sequential(nn.Linear(embedding_dim, projection_dim), nn.Linear(projection_dim, (qk_dim + v_dim) * n_heads)) # Grouped matrix for the heads and k and v

        if causal:
            mask = torch.zeros(max_context, max_context)
            self.register_buffer("mask_array", mask.masked_fill(torch.tril(torch.ones(max_context, max_context, dtype=torch.bool)), float("-inf")))

        self.reproject = nn.Linear(n_heads * v_dim, embedding_dim)

    def forward(self, x):

        x = x[:, -self.max_context:, :]
        batch_dim, seq_len, embed_dim = x.shape
        q = self.q(x) # batch, seq, n_heads * qk_dim

        kv = self.kv(x) # batch, seq, n_heads * (qk_dim + v_dim)
        k, v = torch.split(kv, [self.qk_dim * self.n_heads, self.v_dim * self.n_heads], dim=-1) # batch, seq, n_heads * qk_dim; batch, seq, n_heads * v_dim

        # Use view to prevent copying
        q = self.qk_rope(q.view(batch_dim, seq_len, self.n_heads, self.qk_dim).transpose(1, 2))  # batch, n_heads, seq, qk_dim
        k = self.qk_rope(k.view(batch_dim, seq_len, self.n_heads, self.qk_dim).transpose(1,2)) # batch, n_heads, seq, qk_dim
        v = v.view(batch_dim, seq_len, self.n_heads, self.v_dim).transpose(1,2) #self.v_rope(v.view(batch_dim, seq_len, self.n_heads, self.v_dim).transpose(1,2)) # batch, n_heads, seq, v_dim

        if self.sdpa:
            attn_output = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal, dropout_p=self.dropout)
        else:

            attn_scores = q @ k.transpose(2,3) / self.sqrt_qk_dim  # batch, n_heads, seq, seq

            if self.causal:
                attn_scores += self.mask_array[-seq_len:, -seq_len:]

            attn_probs = torch.softmax(attn_scores, dim=-1)  # batch, n_heads, seq, seq
            if self.dropout > 0 and self.training:
                attn_probs = self.dropout_layer(attn_probs)
            attn_output = attn_probs @ v  # batch, n_heads, seq, v_dim

        #TODO: this reshape copies data. view would be better if I can get the shapes to work.
        return self.reproject(attn_output.transpose(1, 2).reshape(batch_dim, seq_len,
                                                         self.v_dim * self.n_heads))  # batch, seq, embedding_dim

In [55]:
test = MaskedMultiHeadLatentRoPESelfAttention().to('cpu')
x = torch.rand(10, 100, 384).to('cpu')
%timeit test.forward(x)
%timeit test.forward(x)

6.8 ms ± 205 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
7.17 ms ± 626 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [56]:
test = MaskedMultiHeadLatentRoPESelfAttention().to(device)
x = torch.rand(10, 100, 384).to(device)
%timeit test.forward(x)
%timeit test.forward(x)

3.05 ms ± 1.61 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
2.76 ms ± 123 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


## Final code

In [3]:
class RoPETransform(nn.Module):
    def __init__(self, max_seq_len, dim, base=10000):
        super().__init__()
        m_values = torch.arange(max_seq_len, requires_grad=False)
        assert dim % 2 == 0, "RoPE requires even dimension"

        theta_values = torch.pow(base, -2 * torch.arange(dim//2, requires_grad=False) / dim)
        trig_args = torch.outer(m_values, theta_values) # max_seq_len, dim
        # I need to wrap these in a buffer or parameter so that they get moved to the correct device when the model is moved.
        cos_vals = torch.cos(trig_args)
        sin_vals = torch.sin(trig_args)# multiply by alternation -1 +1s for correct rotation application
        self.register_buffer("cos_vals", cos_vals, persistent=False)
        self.register_buffer("sin_vals", sin_vals, persistent=False)

    def forward(self, x):
        batch_dim, n_heads, seq, qk_dim = x.shape
        # x is batch, n_heads, seq, qk_dim
        x1, x2 = x[...,::2], x[...,1::2]
        cos = self.cos_vals[-seq:].to(dtype=x.dtype, device=x.device)
        sin = self.sin_vals[-seq:].to(dtype=x.dtype, device=x.device)
        x_rotated = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2).contiguous()

        return x_rotated

class MaskedMultiHeadLatentRoPESelfAttention(nn.Module):
    def __init__(self, embedding_dim=384, projection_dim=128, qk_dim=64, v_dim=64, causal=True, max_context=100, n_heads=6, sdpa=False, dropout=0.1):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.qk_dim = qk_dim
        self.v_dim = v_dim
        self.projection_dim = projection_dim
        self.max_context = max_context
        self.causal = causal
        self.n_heads = n_heads
        self.sdpa = sdpa
        self.dropout = dropout
        self.dropout_layer = nn.Dropout(dropout)


        self.inv_sqrt_qk_dim = 1/qk_dim ** 0.5

        self.qk_rope = RoPETransform(max_context, qk_dim)
        #self.v_rope = RoPETransform(max_context, v_dim)

        # Want the total number of parameters for kv to be = to the number of parameters for q.
        # q_params = d_embed * d_qk * n_heads
        # kv_params = d_embed * d_proj + d_proj * (d_qk + d_v) * n_heads
        # In the previous layers, I speparated d_qk and dv, letting n_heads * d_v = d_embed. This means that the concatenation of all heads injects a full embedding's worth of information
        # After reading a bit more, it might make sense for d_v = d_qk = d_embed / n_heads. This means that each attention head attends to 1/n_heads worth of embedding, then also injects ~ 1/n_heads embeddings worth of information. Nice and neat. Clarification needed on the empirically optimal choice of these dimensions, as well as the relative balance of parameter counts between q, k, and v.

        # Taking d_v = d_qk, and setting q_params = kv_params, we get:
        # d_embed * d_qk * n_heads = d_embed * d_proj + d_proj * 2 * d_qk * n_heads
        # d_proj = d_embed * d_qk * n_heads / (d_embed + 2 * d_qk * n_heads)
        # if d_qk = d_embed / n_heads:
        # d_proj = d_embed / 3

        # For the default parameters, let embedding_dim = 384, n_heads = 6
        # d_qk = 384 / 6 = 64
        # d_v = d_qk = 384 / 6 = 64
        # d_proj = 384 / 3 = 128

        self.q = nn.Linear(embedding_dim, qk_dim * n_heads) # Grouped matrix for the heads
        self.kv = nn.Sequential(nn.Linear(embedding_dim, projection_dim), nn.Linear(projection_dim, (qk_dim + v_dim) * n_heads)) # Grouped matrix for the heads and k and v

        if causal:
            mask = torch.zeros(max_context, max_context)
            self.register_buffer("mask_array", mask.masked_fill(~torch.tril(torch.ones(max_context, max_context, dtype=torch.bool)), float("-inf")))

        self.reproject = nn.Linear(n_heads * v_dim, embedding_dim)

    def forward(self, x):

        x = x[:, -self.max_context:, :]
        batch_dim, seq_len, embed_dim = x.shape
        q = self.q(x) # batch, seq, n_heads * qk_dim

        kv = self.kv(x) # batch, seq, n_heads * (qk_dim + v_dim)
        k, v = torch.split(kv, [self.qk_dim * self.n_heads, self.v_dim * self.n_heads], dim=-1) # batch, seq, n_heads * qk_dim; batch, seq, n_heads * v_dim

        # Use view to prevent copying
        q = self.qk_rope(q.view(batch_dim, seq_len, self.n_heads, self.qk_dim).transpose(1, 2))  # batch, n_heads, seq, qk_dim
        k = self.qk_rope(k.view(batch_dim, seq_len, self.n_heads, self.qk_dim).transpose(1,2)) # batch, n_heads, seq, qk_dim
        v = v.view(batch_dim, seq_len, self.n_heads, self.v_dim).transpose(1,2) #self.v_rope(v.view(batch_dim, seq_len, self.n_heads, self.v_dim).transpose(1,2)) # batch, n_heads, seq, v_dim

        if self.sdpa:
            attn_output = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal, dropout_p=self.dropout)
        else:

            attn_scores = q @ k.transpose(2,3) * self.inv_sqrt_qk_dim  # batch, n_heads, seq, seq

            if self.causal:
                attn_scores += self.mask_array[-seq_len:, -seq_len:]

            attn_probs = torch.softmax(attn_scores, dim=-1)  # batch, n_heads, seq, seq
            if self.dropout > 0 and self.training:
                attn_probs = self.dropout_layer(attn_probs)
            attn_output = attn_probs @ v  # batch, n_heads, seq, v_dim

        #TODO: this reshape copies data. view would be better if I can get the shapes to work.
        return self.reproject(attn_output.transpose(1, 2).reshape(batch_dim, seq_len,
                                                         self.v_dim * self.n_heads))  # batch, seq, embedding_dim

class TransformerLayer(nn.Module):
    def __init__(self, embedding_dim=384, projection_dim=128, qk_dim=64, v_dim=64, feedforward_dim=1536, causal=True, max_context=100, n_heads=6, norm=nn.RMSNorm, norm_kwargs=None, attn_dropout=0.1, ffn_dropout=0.1, prob_dropout=0.1):
        super().__init__()
        if norm_kwargs is None:
            norm_kwargs = {}
        self.attn_norm = norm(embedding_dim, **norm_kwargs)
        self.ffn_norm = norm(embedding_dim, **norm_kwargs)

        self.attn_dropout = nn.Dropout(attn_dropout)
        self.ffn_dropout = nn.Dropout(ffn_dropout)

        self.attention = MaskedMultiHeadLatentRoPESelfAttention(embedding_dim=embedding_dim, projection_dim=projection_dim, qk_dim=qk_dim, v_dim=v_dim, causal=causal, max_context=max_context, n_heads=n_heads, dropout=prob_dropout)

        self.ffn = nn.Sequential(nn.Linear(embedding_dim, feedforward_dim), nn.GELU(), nn.Dropout(ffn_dropout), nn.Linear(feedforward_dim, embedding_dim))

    def forward(self, x):
        x = x + self.attn_dropout(self.attn_norm(self.attention(x)))
        x = x + self.ffn_dropout(self.ffn_norm(self.ffn(x)))
        return x

class Transformer(nn.Module):
    def __init__(self, num_layers, vocab_size=10000, embedding_dim=384, projection_dim=128, qk_dim=64, v_dim=64, feedforward_dim=1536, causal=True, max_context=100, n_heads=6, norm=nn.RMSNorm, norm_kwargs=None, attn_dropout=0.1, ffn_dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.transformer_stack = nn.Sequential(*[TransformerLayer(embedding_dim=embedding_dim, projection_dim=projection_dim, qk_dim=qk_dim, v_dim=v_dim, feedforward_dim=feedforward_dim, causal=causal, max_context=max_context, n_heads=n_heads, norm=norm, norm_kwargs=norm_kwargs, attn_dropout=attn_dropout, ffn_dropout=ffn_dropout) for _ in range(num_layers)])


        self.final_norm = norm(embedding_dim, **{} if norm_kwargs is None else norm_kwargs)

    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer_stack(x)
        x = self.final_norm(x)
        logits = torch.matmul(x, self.embedding.weight.t())
        return logits

torch.set_default_dtype(torch.bfloat16)
model = Transformer(8).to(dtype=torch.bfloat16, device=device)
model.embedding.weight.data = model.embedding.weight.data.to(torch.bfloat16)
x = torch.randint(0, 9999, (256, 100)).to(device)


In [4]:
y = model(x)

In [5]:
y.shape

torch.Size([256, 100, 10000])

In [23]:
def count_parameters(model):
    return sum([p.numel() for p in model.parameters() if p.requires_grad])


In [26]:
count_parameters(model.transformer_stack)

13010944

In [39]:
max_seq_len = 50
dim = 10
m_values = torch.arange(max_seq_len)
assert dim % 2 == 0, "RoPE requires even dimension"

theta_values = torch.repeat_interleave(torch.pow(base, -2 * torch.arange(dim//2) / dim), 2)
trig_args = torch.outer(m_values, theta_values) # max_seq_len, dim
# I need to wrap these in a buffer or parameter so that they get moved to the correct device when the model is moved.
cos_vals = torch.cos(trig_args)
sin_vals = torch.sin(trig_args) * torch.tensor([[-1, 1]]).repeat(max_seq_len, dim//2)
torch.tensor([[-1, 1]]).repeat(max_seq_len, dim//2)

tensor([[-1,  1, -1,  1, -1,  1, -1,  1, -1,  1],
        [-1,  1, -1,  1, -1,  1, -1,  1, -1,  1],
        [-1,  1, -1,  1, -1,  1, -1,  1, -1,  1],
        [-1,  1, -1,  1, -1,  1, -1,  1, -1,  1],
        [-1,  1, -1,  1, -1,  1, -1,  1, -1,  1],
        [-1,  1, -1,  1, -1,  1, -1,  1, -1,  1],
        [-1,  1, -1,  1, -1,  1, -1,  1, -1,  1],
        [-1,  1, -1,  1, -1,  1, -1,  1, -1,  1],
        [-1,  1, -1,  1, -1,  1, -1,  1, -1,  1],
        [-1,  1, -1,  1, -1,  1, -1,  1, -1,  1],
        [-1,  1, -1,  1, -1,  1, -1,  1, -1,  1],
        [-1,  1, -1,  1, -1,  1, -1,  1, -1,  1],
        [-1,  1, -1,  1, -1,  1, -1,  1, -1,  1],
        [-1,  1, -1,  1, -1,  1, -1,  1, -1,  1],
        [-1,  1, -1,  1, -1,  1, -1,  1, -1,  1],
        [-1,  1, -1,  1, -1,  1, -1,  1, -1,  1],
        [-1,  1, -1,  1, -1,  1, -1,  1, -1,  1],
        [-1,  1, -1,  1, -1,  1, -1,  1, -1,  1],
        [-1,  1, -1,  1, -1,  1, -1,  1, -1,  1],
        [-1,  1, -1,  1, -1,  1, -1,  1, -1,  1],


In [9]:
test = MaskedMultiHeadSelfAttention().to(device)
x = x.to(device)
%timeit test.forward(x)
%timeit test.forward(x)

The slowest run took 90.35 times longer than the fastest. This could mean that an intermediate result is being cached.
24.2 ms ± 49.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
641 ms ± 47.3 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [10]:
test = MaskedMultiHeadSDPASelfAttention().to('cpu')
x = x.to('cpu')
%timeit test.forward(x)
%timeit test.forward(x)

2.29 s ± 693 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
1.77 s ± 429 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [11]:
test = MaskedMultiHeadSDPASelfAttention().to(device)
x = x.to(device)
%timeit test.forward(x)
%timeit test.forward(x)

674 ms ± 23.7 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
699 ms ± 29.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
test = torch.compile(MaskedMultiHeadSelfAttention()).to('cpu')
x = x.to('cpu')
%timeit test.forward(x)
%timeit test.forward(x)
%timeit test.forward(x)
%timeit test.forward(x)

In [27]:
test = torch.compile(MaskedMultiHeadSelfAttention()).to(device)
x = x.to(device)
%timeit test.forward(x)
%timeit test.forward(x)
%timeit test.forward(x)
%timeit test.forward(x)

247 μs ± 25.1 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)
40.5 ms ± 259 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
40.9 ms ± 245 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
40.9 ms ± 167 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
test = torch.compile(MaskedMultiHeadSPDASelfAttention()).to('cpu')
x = x.to('cpu')
%timeit test.forward(x)
%timeit test.forward(x)
%timeit test.forward(x)
%timeit test.forward(x)
test = torch.compile(MaskedMultiHeadSPDASelfAttention()).to(device)
x = x.to(device)
%timeit test.forward(x)
%timeit test.forward(x)
%timeit test.forward(x)
%timeit test.forward(x)

In [4]:
q = nn.Linear(1000, 100)(torch.randn(20, 3,1000)) # batch, seq, dim
k = nn.Linear(1000, 100)(torch.randn(20, 3,1000))
v = nn.Linear(1000, 1000)(torch.randn(20, 3,1000))

In [25]:
((q @ k.mT) @ v).shape

torch.Size([20, 3, 1000])

In [5]:
q.shape # batch, seq, embed_dim

torch.Size([20, 3, 100])

In [7]:
(q.reshape(20, 3, 5, 20) @ k.reshape(20, 3, 5, 20).mT).shape

torch.Size([20, 3, 5, 5])

In [8]:
(q @ k.mT).shape

torch.Size([20, 3, 3])